# 01 — Online Retail II transaction cleaning

**Purpose:** combine the two UCI Online Retail II workbook sheets and create the cleaned transaction file used by the later notebooks.

**Source:** [UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail), covering 1 December 2009 through 9 December 2011. The dataset is CC BY 4.0; retain the provider attribution when redistributing data or derived material.

**Input:** `../data/raw/online_retail_II.xlsx`  
**Outputs:** `../data/raw/online_retail_II.csv` and `../data/processed/clean_transactions.csv`

Run Jupyter with the working directory set to this `notebooks/` folder so the relative paths in all six notebooks resolve correctly. The large source dataset is intentionally not included in the repository or submission ZIP; download it from UCI and place the workbook at the input path above.

The cleaning rules below remove cancelled invoices, non-positive quantity/price rows, and non-product stock codes. Customer ID is removed because it is not used by the model pipeline.

In [5]:
import pandas as pd

path = "../data/raw/online_retail_II.xlsx"

sheets = pd.read_excel(path, sheet_name=None)

print("Available sheets:", sheets.keys())

for name, sheet_df in sheets.items():
    print(name, sheet_df.shape)

print("\nAll Excel sheets loaded successfully.")

Available sheets: dict_keys(['Year 2009-2010', 'Year 2010-2011'])
Year 2009-2010 (525461, 8)
Year 2010-2011 (541910, 8)

All Excel sheets loaded successfully.


In [6]:
# Combine both Excel sheets into one transaction dataframe

df = pd.concat(list(sheets.values()), ignore_index=True)

print("Combined dataset shape:", df.shape)
print("Total transactions:", len(df))

assert len(sheets) == 2
assert df.shape == (1_067_371, 8)

df.to_csv("../data/raw/online_retail_II.csv", index=False)

print("\nBoth sheets successfully combined.")

Combined dataset shape: (1067371, 8)
Total transactions: 1067371

Both sheets successfully combined.


In [7]:
df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [8]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype         
---  ------       --------------    -----         
 0   Invoice      1067371 non-null  object        
 1   StockCode    1067371 non-null  object        
 2   Description  1062989 non-null  object        
 3   Quantity     1067371 non-null  int64         
 4   InvoiceDate  1067371 non-null  datetime64[us]
 5   Price        1067371 non-null  float64       
 6   Customer ID  824364 non-null   float64       
 7   Country      1067371 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 78.8+ MB


In [9]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [10]:
df.describe()

,Quantity,InvoiceDate,Price,Customer ID
count,1.067371e+06,1067371,1.067371e+06,824364.000000
mean,9.938898e+00,2011-01-02 21:13:55.394029,4.649388e+00,15324.638504
min,-8.099500e+04,2009-12-01 07:45:00,-5.359436e+04,12346.000000
25%,1.000000e+00,2010-07-09 09:46:00,1.250000e+00,13975.000000
50%,3.000000e+00,2010-12-07 15:28:00,2.100000e+00,15255.000000
75%,1.000000e+01,2011-07-22 10:23:00,4.150000e+00,16797.000000
max,8.099500e+04,2011-12-09 12:50:00,3.897000e+04,18287.000000
std,1.727058e+02,NaN,1.235531e+02,1697.464450


In [11]:
# These counts goes in Report
print((df["Quantity"] < 0).sum(), "rows with negative Quantity")
print((df["Price"] <= 0).sum(), "rows with Price <= 0")
print(df["Invoice"].astype(str).str.startswith("C").sum(), "rows associated with cancelled invoices")

22950 rows with negative Quantity
6207 rows with Price <= 0
19494 rows associated with cancelled invoices


In [12]:
clean = df.copy()
print(f"Start: {len(clean):,} rows")

clean = clean[~clean["Invoice"].astype(str).str.startswith("C")]
print(f"After removing cancellations: {len(clean):,} rows")

Start: 1,067,371 rows
After removing cancellations: 1,047,877 rows


In [13]:
clean = clean[(clean["Quantity"] > 0) & (clean["Price"] > 0)]
print(f"After removing Quantity<=0 or Price<=0: {len(clean):,} rows")

After removing Quantity<=0 or Price<=0: 1,041,670 rows


In [14]:
clean["StockCode"] = clean["StockCode"].astype(str).str.upper().str.strip()
is_product = clean["StockCode"].str.match(r"^\d{5}[A-Z]*$")
print(clean.loc[~is_product, "StockCode"].value_counts().head(20))

StockCode
POST            1886
DOT             1436
M                881
C2               272
ADJUST            36
BANK CHARGES      35
DCGS0058          30
GIFT_0001_20      26
GIFT_0001_30      24
DCGSSGIRL         23
DCGSSBOY          21
PADS              17
DCGS0076          15
GIFT_0001_10      14
DCGS0003          13
TEST001            9
GIFT_0001_50       6
DCGS0069           5
D                  5
GIFT_0001_40       5
Name: count, dtype: int64


In [20]:
print("Current shape:", clean.shape)
print("Current columns:", clean.columns.tolist())
print("Customer ID exists:", "Customer ID" in clean.columns)

Current shape: (1036877, 7)
Current columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Country']
Customer ID exists: False


In [21]:
# Remove non-product codes and drop the unused Customer ID column

clean = clean.loc[is_product].copy()

print(f"After removing non-product codes: {len(clean):,} rows")

# Remove Customer ID only if it still exists.
# This makes the cell safe to execute more than once.
clean = clean.drop(columns=["Customer ID"], errors="ignore")

print("\nFinal dataframe shape:", clean.shape)

print("\nMissing values:")
print(clean.isna().sum())

# Save cleaned transactions
clean.to_csv("../data/processed/clean_transactions.csv", index=False)

print("\nCleaned dataset saved successfully.")

After removing non-product codes: 1,036,877 rows

Final dataframe shape: (1036877, 7)

Missing values:
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Country        0
dtype: int64

Cleaned dataset saved successfully.


## Cleaning validation

The checks below verify the rules applied above. They are data-quality assertions, not additional transformations. The expected cleaned row count for the supplied UCI workbook is 1,036,877.

In [16]:
assert (clean["Quantity"] > 0).all()
assert (clean["Price"] > 0).all()
assert not clean["Invoice"].astype(str).str.startswith("C").any()
assert clean["StockCode"].astype(str).str.match(r"^\d{5}[A-Z]*$").all()
assert not clean.isna().any().any()

print(f"Validated cleaned rows: {len(clean):,}")
print(f"Unique products: {clean['StockCode'].nunique():,}")
print("All cleaning assertions passed.")

Validated cleaned rows: 1,036,877
Unique products: 4,707
All cleaning assertions passed.
